In [ ]:
import os
MODE = 'superlevel'         
tag  = 'super'               
pd_dir = f'/{direct}/result_mmCen/pd_super/'
os.makedirs(pd_dir, exist_ok=True)
minv=-0.5
maxv=0.9


In [ ]:
#cell pd
def peristent_homology_sublevel_cubic_entropy(
    df,#u_merge, s, c_merge
    region,
    samplename,#cell1_U
    target,
    minv,
    maxv,
    save_path1,
    mode):#'/mnt/g/cell/20250707XY/result/pd/'
    '''Calculate sublevel set cubical homology persistence for an image
    
    Takes the idiagram file and pulls out the birth,death persistence intervals and persistent diagrams
    for two dimensions (0,1). 
    Saves the persistence intervals as csv.
    Optionally plots the persistence diagrams and saves as svg.

    Args:
        image (numpy array): image values along which to take the filtration
        filename (string): filename for image as string
        save_path (string): location to save necessary files as string
        plot_persistence_diagrams (bool, optional): If True saves plots of the 
            persistence diagram. Defaults to False.
    '''
    import numpy as np
    import matplotlib.pyplot as plt
    import pandas as pda
    import homcloud.interface as hc

        
    #import data
    whole = df[df['label1']==samplename]
    test1 = whole #whole brain
    #for picture
    p1 = test1.pivot_table(index=['y'], columns=['x'],values=['syn'], aggfunc='sum') #y_bin x_bin syn_bin
      
    #pivotTest = p.fillna(0)
    np.savetxt(save_path1+samplename+"pivot"+region+"_pic.txt", p1, delimiter='\t')

    #import image
    pict1 = np.loadtxt(save_path1+samplename+"pivot"+region+"_pic.txt")
    #plt.figure()
    plt.imshow(pict1, cmap='gray', origin='lower')
    plt.title(f'{samplename}_{region}')
    plt.axis('off')  # Remove x and y axes labels and ticks
    plt.savefig(save_path1+samplename+region+".png")
    plt.close()
    #plt.imshow(pict,"gray")
    #plt.show()

    #for calculation
    p = test1.pivot_table(index=['y'], columns=['x'],values=[target], aggfunc='sum') #y_bin x_bin
    vals = p.values
    inside = ~np.isnan(vals) 
    #pivotTest = p.fillna(0)
    np.savetxt(save_path1+samplename+"pivot"+region+".txt", p, delimiter='\t')

    #import table
    pict = np.loadtxt(save_path1+samplename+"pivot"+region+".txt")
    fill = np.nanmax(vals) if mode == "sublevel" else np.nanmin(vals)
    pict = np.where(inside, vals, fill)
    
    #calculate sublevel set cubical homology persistence for each image
    hc.PDList.from_bitmap_levelset(pict, mode, save_to=save_path1+samplename+"grayscale"+region+".pdgm")

    # for dimensions 0 and 1, extract the births and deaths
    #pds = [hc.PDList(save_path1+"grayscale"+region+".pdgm").dth_diagram(i) for i in range(0,2)]    

    for i in range(0,2):
        pds = hc.PDList(save_path1+samplename+"grayscale"+region+".pdgm")
        pd = pds.dth_diagram(i)
        
        # pd and pairs
        fig, ax = plt.subplots(figsize=(6.5, 5.8))  # 手动创建一个 Figure 和 Axes 98
        pd.histogram(x_bins=64).plot(ax=ax,colorbar={"type": "log"})#x_bins=64

        # 修改 x 和 y 轴刻度字体大小
        ax.tick_params(axis='x', labelsize=22)  # x 轴刻度字体大小 30
        ax.tick_params(axis='y', labelsize=22)  # y 轴刻度字体大小

        # 修改 x 和 y 轴标签字体大小
        ax.set_xlabel(ax.get_xlabel(), fontsize=18)  # x 轴标签字体大小
        ax.set_ylabel(ax.get_ylabel(), fontsize=18)  # y 轴标签字体大小

        #修改下x,y范围
        ax.set_xlim([minv, maxv])
        ax.set_ylim([minv, maxv])

        # 画对角线，使其延伸至整个 x, y 轴范围
        ax.plot([minv,maxv], [minv, maxv], color="black", linestyle="-", linewidth=1)


        # 获取 colorbar 对象并设置字体大小
        cbar = ax.collections[-1].colorbar
        cbar.ax.tick_params(labelsize=18)
        
        plt.savefig(save_path1+"d"+str(i)+"_"+samplename+region+"_64.png")#_64
        plt.close()
        

        
        # === Direct PD statistics ===
        
        births_raw = np.asarray(pd.births, dtype=float)
        deaths_raw = np.asarray(pd.deaths, dtype=float)
        
        # 统计原始区间数量
        n_total = len(births_raw)
        
        # 只保留 birth 和 death 都有限的区间
        finite = np.isfinite(births_raw) & np.isfinite(deaths_raw)
        
        births_cell = births_raw[finite]
        deaths_cell = deaths_raw[finite]
        
        n_finite = len(births_cell)
        n_infinite = n_total - n_finite
        
        # 修改：按 mode 判断方向（sublevel: death>=birth；superlevel: birth>=death）
        # 修改：原来只检查 death < birth，superlevel 下一定会报错
        if mode == 'sublevel':
            if np.any(deaths_cell < births_cell):
                raise ValueError("Found death < birth. Please check the PD data.")
        else:
            if np.any(deaths_cell > births_cell):
                raise ValueError("Found death > birth. Please check the PD data.")
        
        # Persistence
        # 修改：superlevel 的 persistence = birth - death（原来固定为 death - birth，会变成负数）
        if mode == 'sublevel':
            persistence_cell = deaths_cell - births_cell
        else:
            persistence_cell = births_cell - deaths_cell
        
        # === Birth statistics ===
        birth_mean = np.mean(births_cell)
        birth_median = np.median(births_cell)
        birth_q25 = np.percentile(births_cell, 25)
        birth_q75 = np.percentile(births_cell, 75)
        birth_iqr = birth_q75 - birth_q25
        birth_variance = np.var(births_cell)
        birth_std = np.std(births_cell)
        
        # === Death statistics ===
        death_mean = np.mean(deaths_cell)
        death_median = np.median(deaths_cell)
        death_q25 = np.percentile(deaths_cell, 25)
        death_q75 = np.percentile(deaths_cell, 75)
        death_iqr = death_q75 - death_q25
        death_variance = np.var(deaths_cell)
        death_std = np.std(deaths_cell)
        
        # === Persistence statistics ===
        persistence_mean = np.mean(persistence_cell)
        persistence_median = np.median(persistence_cell)
        persistence_q25 = np.percentile(persistence_cell, 25)
        persistence_q75 = np.percentile(persistence_cell, 75)
        persistence_iqr = persistence_q75 - persistence_q25
        persistence_variance = np.var(persistence_cell)
        persistence_std = np.std(persistence_cell)
        
        # 99th percentile
        persistence_99th = np.percentile(persistence_cell, 99)
        
        # Total persistence
        total_persistence = np.sum(persistence_cell)
        
        # Persistent entropy
        if total_persistence > 0:
            probabilities = persistence_cell / total_persistence
            probabilities = probabilities[probabilities > 0]
            persistent_entropy = -np.sum(
                probabilities * np.log(probabilities)
            )
        else:
            persistent_entropy = np.nan
        
        # === Result ===
        result = pda.DataFrame({
            'sample': [samplename],
            'dim': [i],
            'group': [region],
        
            # Number of intervals
            'n_total': [n_total],
            'n_finite': [n_finite],
            'n_infinite': [n_infinite],
        
            # Birth
            'birth_mean': [birth_mean],
            'birth_median': [birth_median],
            'birth_q25': [birth_q25],
            'birth_q75': [birth_q75],
            'birth_iqr': [birth_iqr],
            'birth_variance': [birth_variance],
            'birth_std': [birth_std],
        
            # Death
            'death_mean': [death_mean],
            'death_median': [death_median],
            'death_q25': [death_q25],
            'death_q75': [death_q75],
            'death_iqr': [death_iqr],
            'death_variance': [death_variance],
            'death_std': [death_std],
        
            # Persistence
            'persistence_mean': [persistence_mean],
            'persistence_median': [persistence_median],
            'persistence_q25': [persistence_q25],
            'persistence_q75': [persistence_q75],
            'persistence_iqr': [persistence_iqr],
            'persistence_variance': [persistence_variance],
            'persistence_std': [persistence_std],
            'persistence_99th_percentile': [persistence_99th],
        
            # Additional PH statistics
            'total_persistence': [total_persistence],
            'persistent_entropy': [persistent_entropy]
        })
        
        result.to_csv(
            save_path1 + f"d{i}_{samplename}{region}.csv",
            index=False
        )

        
        del pds


In [ ]:
target='mmCenbin_syn'
def homo_group(df, region):
    for label in df['label1'].unique():  
        peristent_homology_sublevel_cubic_entropy(
            df,#u_merge, s, c_merge
            region,
            samplename=label,#1-1
            target=target,
            minv=minv,
            maxv=maxv,
            save_path1=pd_dir,   
            mode=MODE)  

homo_group(s, 'SNCA')   # s 组
homo_group(u, 'UBL3')   # u 组
homo_group(c, 'CAAX')   # c 组